In [1]:
import pandas as pd

# Đọc dữ liệu từ file parquet
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")

# Chuyển đổi timestamp sang số (giây)
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

# Sắp xếp tĩnh trực tiếp theo thời gian
train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

print("Dữ liệu đã được load và sắp xếp theo thời gian thành công!")


Dữ liệu đã được load và sắp xếp theo thời gian thành công!


In [16]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score
from tqdm.notebook import tqdm
import gc

# =================================================================================
# 2. CẤU TRÚC ĐỒ THỊ FLOW-AS-NODE (TỐI ƯU HÓA VECTORIZATION)
# =================================================================================
def build_flow_as_node_snapshots(df, feature_cols, chunk_size=5000, max_k=10):
    """
    Thuật toán biểu diễn Đồ thị Flow-as-Node.
    Mỗi bản ghi Flow đóng vai trò là một Node.
    Hai Node được kết nối (Edge) nếu chúng chia sẻ cùng Source/Destination IP.
    
    TỐI ƯU HOÁ (Chống đứng máy): 
    Thay vì vòng lặp Python O(N^2) làm nổ RAM/CPU khi có một IP tạo 5000 kết nối DDoS. 
    Chúng ta dùng NumPy và thuật toán trượt K-láng giềng (K-neighbors) để ép giới hạn tỷ lệ cạnh xuống O(N).
    """
    print(f"Xây dựng Graph với Node=Flow (Chunk size: {chunk_size})...")
    snapshots = []
    
    # BỎ TRỘN NGẪU NHIÊN: Giữ nguyên thứ tự thời gian!
    # Các dòng dữ liệu đã được sắp xếp theo `timestamp_num` từ Cell 1.
    # Khi không trộn, các flows đứng cạnh nhau của cùng 1 IP sẽ là các hành vi ĐỒNG THỜI (chung 1 đợt tấn công).
    df_chunked = df.reset_index(drop=True)
    num_chunks = int(np.ceil(len(df_chunked) / chunk_size))
    
    def get_edges_for_group(indices, base_idx, max_K):
        """Hàm nối kết nối nội bộ nhóm cực nhanh bằng mảng trượt NumPy"""
        arr = np.array(indices) - base_idx
        n = len(arr)
        if n < 2: return [], []
        
        edges_u, edges_v = [], []
        # Nối kết mỗi Flow với tối đa K Flow khác trong cùng nhóm
        # Biến O(N^2) thành O(N*K) và vector hóa
        for offset in range(1, min(n, max_K + 1)):
            u = arr[:-offset]
            v = arr[offset:]
            edges_u.extend(u)
            edges_v.extend(v)
            # Nối hai chiều
            edges_u.extend(v)
            edges_v.extend(u)
        return edges_u, edges_v

    for i in tqdm(range(num_chunks), desc="Tạo Flow-Node Graphs"):
        chunk = df_chunked.iloc[i * chunk_size : (i+1) * chunk_size].copy()
        if len(chunk) == 0: continue
            
        # 1. Trích xuất Node Features
        x = torch.tensor(chunk[feature_cols].values, dtype=torch.float32)
        y = torch.tensor(chunk['label'].values, dtype=torch.long)
        
        # 2. Xây dựng Edges Siêu Tốc
        src_groups = chunk.groupby('src_ip').groups
        dst_groups = chunk.groupby('dst_ip').groups
        
        base_idx = i * chunk_size
        edges_source = []
        edges_target = []
        
        # Nhóm Source IP
        for ip, indices in src_groups.items():
            u, v = get_edges_for_group(indices, base_idx, max_k)
            edges_source.extend(u)
            edges_target.extend(v)

        # Nhóm Dst IP
        for ip, indices in dst_groups.items():
            u, v = get_edges_for_group(indices, base_idx, max_k)
            edges_source.extend(u)
            edges_target.extend(v)
                            
        # Loại bỏ các cặp trùng lặp (ví dụ 2 dòng có chung CẢ src_ip VÀ dst_ip)
        if len(edges_source) > 0:
            e_idx = np.array([edges_source, edges_target])
            e_idx = np.unique(e_idx, axis=1) # NumPy unique chạy cực nhanh trong backend C
            edge_index = torch.tensor(e_idx, dtype=torch.long)
        else:
            edge_index = torch.empty((2, 0), dtype=torch.long)
            
        graph = Data(x=x, edge_index=edge_index, y=y)
        snapshots.append(graph)
        
    print(f"Đã tạo xong {len(snapshots)} Flow-Node Graphs siêu tốc!")
    return snapshots

# =================================================================================
# 3. KIẾN TRÚC MÔ HÌNH (GCN CHO NODE CLASSIFICATION)
# =================================================================================
class FlowNode_GCN(nn.Module):
    def __init__(self, num_node_features, num_classes):
        super().__init__()
        # Dùng LayerNorm thay cho BatchNorm (SỐNG CÒN cho dữ liệu Time-Series/Chunk)
        self.input_norm = nn.LayerNorm(num_node_features)
        
        self.linear_in = nn.Linear(num_node_features, 128)
        
        # Đổi GCNConv thành SAGEConv: Hoạt động ưu việt hơn với Graph kết nối trượt
        self.conv1 = SAGEConv(128, 256)
        self.norm1 = nn.LayerNorm(256)
        
        self.conv2 = SAGEConv(256, 128)
        self.norm2 = nn.LayerNorm(128)
        
        # Giảm dropout
        self.dropout = nn.Dropout(p=0.2)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x, edge_index):
        x = self.input_norm(x)
        x = self.linear_in(x)
        
        # Layer 1
        x = self.conv1(x, edge_index)
        x = self.norm1(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        # Layer 2
        x = self.conv2(x, edge_index)
        x = self.norm2(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        return self.classifier(x)

In [ ]:
import gc
import torch
import numpy as np

# Số lượng Flow (Nodes) trên mỗi Graph Chunking
CHUNK_SIZE = 10000

# Xác định danh sách các cột Feature của Flow 
# Dùng select_dtypes để TỰ ĐỘNG CHỈ LẤY CÁC CỘT DẠNG SỐ (floats, ints), tránh đưa string, ip_address vào Tensor
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
ignored_cols = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']
feature_cols = [col for col in numeric_cols if col not in ignored_cols]

print(f"Số lượng Node Features: {len(feature_cols)}")

print("--- 1. TẠO ĐỒ THỊ TẬP TRAIN ---")
train_snapshots = build_flow_as_node_snapshots(train_df, feature_cols=feature_cols, chunk_size=CHUNK_SIZE)

print("\n--- 2. TẠO ĐỒ THỊ TẬP VALID ---")
valid_snapshots = build_flow_as_node_snapshots(valid_df, feature_cols=feature_cols, chunk_size=CHUNK_SIZE)

print("\n--- 3. TẠO ĐỒ THỊ TẬP TEST ---")
test_snapshots = build_flow_as_node_snapshots(test_df, feature_cols=feature_cols, chunk_size=CHUNK_SIZE)

# Giải phóng RAM
gc.collect()

# =================================================================================
# KIỂM TRA THUỘC TÍNH CỦA ĐỒ THỊ (GRAPH DIAGNOSTICS)
# =================================================================================
print("\n=======================================================")
print("KIỂM TRA THUỘC TÍNH CỦA CÁC FLOW-NODE GRAPHS VỪA TẠO")
print("=======================================================")

def inspect_snapshots(name, snapshots):
    num_graphs = len(snapshots)
    total_nodes = sum(g.num_nodes for g in snapshots)
    total_edges = sum(g.num_edges for g in snapshots)
    
    if num_graphs == 0:
        print(f"- {name}: 0 đồ thị!")
        return
        
    max_node_graph = max(snapshots, key=lambda g: g.num_nodes)
    
    print(f"\n[{name}] Tổng số Nhóm Graph: {num_graphs:,} | Tổng số Nodes (Flows): {total_nodes:,} | Tổng số Edges (Connections): {total_edges:,}")
    print(f" -> Kích thước Cấu trúc (edge_index shape): {max_node_graph.edge_index.shape}")
    print(f" -> Kích thước Features (x shape): {max_node_graph.x.shape}")
    print(f" -> Kích thước Nhãn (y shape): {max_node_graph.y.shape}")
    
    all_labels = torch.cat([g.y for g in snapshots])
    classes, counts = torch.unique(all_labels, return_counts=True)
    
    print(f" -> Phân phối nhãn (Label Distribution) trong {name}:")
    for c, count in zip(classes.tolist(), counts.tolist()):
        pct = count / len(all_labels) * 100
        print(f"    - Lớp {c}: {count:>8,} nodes ({pct:>6.2f}%)")

inspect_snapshots("TRAIN", train_snapshots)
inspect_snapshots("VALID", valid_snapshots)
inspect_snapshots("TEST", test_snapshots)

In [20]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# =================================================================================
# BƯỚC 1: XÁC ĐỊNH THIẾT BỊ TÍNH TOÁN VÀ ĐẶC TRƯNG CLASES
# =================================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"-> Đang sử dụng thiết bị: {device}")

all_train_labels = torch.cat([g.y.cpu() for g in train_snapshots]).numpy()
unique_classes = np.unique(all_train_labels)
num_classes = len(unique_classes)
num_node_features = train_snapshots[0].x.shape[1] if len(train_snapshots) > 0 else 0
print(f"-> Tổng số nhãn (classes): {num_classes}")
print(f"-> Số features theo Flow-Node: {num_node_features}")

# BẢO VỆ CLASS WEIGHTS
class_weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=all_train_labels)
class_weights = np.nan_to_num(class_weights, nan=1.0, posinf=10.0, neginf=1.0)
# LÀM MỀM TRỌNG SỐ (Smoothing) - Tránh áp đảo/bỏ qua lớp Majority
class_weights = np.power(class_weights, 0.4) 
class_weights = np.clip(class_weights, a_min=0.5, a_max=6.0) 
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

# =================================================================================
# BƯỚC 2: HÀM HUẤN LUYỆN CHO NODE CLASSIFICATION
# =================================================================================
def train_flow_node_gcn(model, train_snapshots, valid_snapshots, device, class_weights, num_epochs=100):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4) 
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)
    
    best_val_loss = float('inf')
    early_stop_count = 0
    patience = 15
    
    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0.0
        all_train_preds, all_train_labels = [], []
        total_train_nodes = sum(g.num_nodes for g in train_snapshots)
        
        for g in train_snapshots:
            if g.num_nodes == 0: continue
            g = g.to(device)
            
            optimizer.zero_grad(set_to_none=True)
            
            logits = model(g.x, g.edge_index)
            
            loss = criterion(logits, g.y)
            loss.backward()
            
            # Clip gradient trực tiếp trên từng đồ thị để tránh Gradient Explosion
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_train_loss += (loss.item() * g.num_nodes)
            preds = logits.argmax(dim=1)
            all_train_preds.extend(preds.cpu().tolist())
            all_train_labels.extend(g.y.cpu().tolist())

        train_f1 = f1_score(all_train_labels, all_train_preds, average='macro', zero_division=0)
        avg_train_loss = total_train_loss / max(1, total_train_nodes)
        
        # --- VALIDATION ---
        model.eval()
        total_val_loss = 0.0
        all_val_preds, all_val_labels = [], []
        total_val_nodes = sum(g.num_nodes for g in valid_snapshots)
        
        with torch.no_grad():
            for g in valid_snapshots:
                if g.num_nodes == 0: continue
                g = g.to(device)
                
                logits = model(g.x, g.edge_index)
                loss = criterion(logits, g.y)
                
                total_val_loss += loss.item() * g.num_nodes
                preds = logits.argmax(dim=1)
                all_val_preds.extend(preds.cpu().tolist())
                all_val_labels.extend(g.y.cpu().tolist())
                    
        val_f1 = f1_score(all_val_labels, all_val_preds, average='macro', zero_division=0)
        avg_val_loss = total_val_loss / max(1, total_val_nodes)
        
        scheduler.step(avg_val_loss)
        print(f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f} (F1 Macro: {train_f1:.4f}) | Val Loss: {avg_val_loss:.4f} (F1 Macro: {val_f1:.4f})")
              
        if avg_val_loss < best_val_loss and not np.isnan(avg_val_loss):
            best_val_loss = avg_val_loss
            early_stop_count = 0
            torch.save(model.state_dict(), "SOTA_DynamicIDS_FlowNode_Best.pth")
        else:
            early_stop_count += 1
            if early_stop_count >= patience:
                print(f"=== Đã kích hoạt Early Stopping tại Epoch {epoch+1}! ===")
                break

# =================================================================================
# BẮT ĐẦU HUẤN LUYỆN
# =================================================================================
model = FlowNode_GCN(num_node_features=num_node_features, num_classes=num_classes).to(device)

print("\n--- BẮT ĐẦU HUẤN LUYỆN (NODE CLASSIFICATION) ---")
train_flow_node_gcn(
    model=model, 
    train_snapshots=train_snapshots, 
    valid_snapshots=valid_snapshots, 
    device=device, 
    class_weights=class_weights_tensor, 
    num_epochs=400
)

# =================================================================================
# KIỂM THỬ TRÊN TẬP TEST VÀ ĐÁNH GIÁ (TESTING)
# =================================================================================
print("\n--- KIỂM THỬ TRÊN TẬP TEST ---")
try:
    model.load_state_dict(torch.load("SOTA_DynamicIDS_FlowNode_Best.pth", map_location=device))
except:
    print("Không tải được file Weight, giữ nguyên Weight hiện tại.")
model.eval()

all_test_preds, all_test_labels = [], []
with torch.no_grad():
    for g in test_snapshots:
        if g.num_nodes == 0: continue
        g = g.to(device)
        
        logits = model(g.x, g.edge_index)
        preds = logits.argmax(dim=1)
        
        all_test_preds.extend(preds.cpu().tolist())
        all_test_labels.extend(g.y.cpu().tolist())

target_names = [f"Lớp {i}" for i in range(num_classes)]
print("\nBÁO CÁO PHÂN LOẠI (CLASSIFICATION REPORT) TRÊN TẬP TEST:")
print(classification_report(all_test_labels, all_test_preds, target_names=target_names, zero_division=0))

-> Đang sử dụng thiết bị: cuda
-> Tổng số nhãn (classes): 11
-> Số features theo Flow-Node: 315

--- BẮT ĐẦU HUẤN LUYỆN (NODE CLASSIFICATION) ---


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 001 | Train Loss: 1.3467 (F1 Macro: 0.2145) | Val Loss: 5.9814 (F1 Macro: 0.0225)
Epoch 002 | Train Loss: 1.5362 (F1 Macro: 0.1369) | Val Loss: 6.4319 (F1 Macro: 0.0225)
Epoch 003 | Train Loss: 1.4971 (F1 Macro: 0.1539) | Val Loss: 5.5933 (F1 Macro: 0.0225)
Epoch 004 | Train Loss: 1.3065 (F1 Macro: 0.1569) | Val Loss: 6.4009 (F1 Macro: 0.0225)
Epoch 005 | Train Loss: 1.2249 (F1 Macro: 0.1750) | Val Loss: 6.7906 (F1 Macro: 0.0225)
Epoch 006 | Train Loss: 1.0425 (F1 Macro: 0.1966) | Val Loss: 6.4689 (F1 Macro: 0.0225)
Epoch 007 | Train Loss: 1.3416 (F1 Macro: 0.1852) | Val Loss: 7.2664 (F1 Macro: 0.0225)
Epoch 008 | Train Loss: 1.2768 (F1 Macro: 0.1838) | Val Loss: 6.0064 (F1 Macro: 0.0225)
Epoch 009 | Train Loss: 1.0134 (F1 Macro: 0.2034) | Val Loss: 6.5990 (F1 Macro: 0.0225)
Epoch 010 | Train Loss: 1.5073 (F1 Macro: 0.1452) | Val Loss: 5.6642 (F1 Macro: 0.0225)
Epoch 011 | Train Loss: 2.2607 (F1 Macro: 0.0825) | Val Loss: 2.1148 (F1 Macro: 0.0225)
Epoch 012 | Train Loss: 1.8777 (

C:\Users\Admin\AppData\Local\Temp\ipykernel_8380\3955513230.py:129: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("SOTA_DynamicIDS_FlowNode_


BÁO CÁO PHÂN LOẠI (CLASSIFICATION REPORT) TRÊN TẬP TEST:
              precision    recall  f1-score   support

       Lớp 0       1.00      1.00      1.00     19846
       Lớp 1       1.00      1.00      1.00    484077
       Lớp 2       0.99      0.80      0.89      2515
       Lớp 3       1.00      0.92      0.96     36253
       Lớp 4       1.00      1.00      1.00     18979
       Lớp 5       1.00      1.00      1.00      2451
       Lớp 6       0.77      0.86      0.81     11847
       Lớp 7       1.00      1.00      1.00    107436
       Lớp 8       0.89      1.00      0.94     16746
       Lớp 9       1.00      0.67      0.80     41523
      Lớp 10       0.58      1.00      0.74     18567

    accuracy                           0.98    760240
   macro avg       0.93      0.93      0.92    760240
weighted avg       0.98      0.98      0.98    760240



In [22]:
print(classification_report(all_test_labels, all_test_preds, target_names=target_names, digits=4))

              precision    recall  f1-score   support

       Lớp 0     0.9987    0.9987    0.9987     19846
       Lớp 1     0.9998    1.0000    0.9999    484077
       Lớp 2     0.9946    0.8024    0.8882      2515
       Lớp 3     0.9998    0.9250    0.9609     36253
       Lớp 4     0.9974    0.9986    0.9980     18979
       Lớp 5     0.9959    0.9996    0.9978      2451
       Lớp 6     0.7677    0.8608    0.8116     11847
       Lớp 7     1.0000    1.0000    1.0000    107436
       Lớp 8     0.8887    0.9996    0.9409     16746
       Lớp 9     1.0000    0.6669    0.8002     41523
      Lớp 10     0.5812    0.9996    0.7350     18567

    accuracy                         0.9753    760240
   macro avg     0.9294    0.9319    0.9210    760240
weighted avg     0.9834    0.9753    0.9760    760240



In [8]:
import torch
from torch_geometric.utils import homophily, degree
import numpy as np

print("=======================================================")
print("KIỂM TRA CHUYÊN SÂU TÍNH CHẤT ĐỒ THỊ (DIAGNOSTICS)")
print("=======================================================")

# Lấy thử 1 graph ngẫu nhiên đại diện trong tập TRAIN (chọn graph lớn nhất để test)
g = max(train_snapshots, key=lambda g: g.num_edges)
g = g.cpu()

# 1. Đo lường tỷ lệ đồng chất (Homophily Ratio)
# Tính tỷ lệ cạnh nối giữa 2 Node CÙNG LỚP so với tổng số cạnh
h_ratio = homophily(g.edge_index, g.y, method='edge')
print(f"1. Homophily Ratio (Tỷ lệ đồng chất): {h_ratio:.4f}")
print("   -> (Homophily < 0.5 nghĩa là đồ thị mang tính dị chất (Heterophilic). Thông tin giữa các class khác nhau bị trộn lẫn gây nhiễu GNN).")

# 2. Phân tích Các cạnh khác Class (Heterophilic Edges)
row, col = g.edge_index
src_labels = g.y[row]
dst_labels = g.y[col]

same_class_edges = (src_labels == dst_labels).sum().item()
diff_class_edges = g.num_edges - same_class_edges
print(f"\n2. Tổng số Edges: {g.num_edges:,}")
print(f"   -> Cạnh cùng lớp (Safe): {same_class_edges:,} ({same_class_edges/g.num_edges*100:.2f}%)")
print(f"   -> Cạnh khác lớp (Poison): {diff_class_edges:,} ({diff_class_edges/g.num_edges*100:.2f}%)")

# 3. Phân phối Bậc (Node Degree)
deg = degree(row, g.num_nodes)
print(f"\n3. Phân phối Node Degree (Số kết nối của 1 Node):")
print(f"   -> Trung bình: {deg.mean().item():.2f}")
print(f"   -> Cao nhất (Hub): {deg.max().item():.2f}")
print(f"   -> Thấp nhất: {deg.min().item():.2f}")
print(f"   -> Số Node bị cô lập (Degree=0): {(deg == 0).sum().item()}")

# 4. Kiểm tra Dải giá trị Đặc trưng đầu vào (Feature Scales)
print(f"\n4. Thống kê giá trị Feature (x) thô:")
print(f"   -> Giá trị Min: {g.x.min().item():.4f}")
print(f"   -> Giá trị Max: {g.x.max().item():.4f}")
print(f"   -> Giá trị Mean: {g.x.mean().item():.4f}")
if g.x.max().item() > 1000 or g.x.min().item() < -1000:
    print("   -> CẢNH BÁO: Dữ liệu chưa qua Standard Scaler hoặc MinMax Scaler, GNN trực tiếp nhân MA TRẬN KHỔNG LỒ sẽ nổ Gradient.")


KIỂM TRA CHUYÊN SÂU TÍNH CHẤT ĐỒ THỊ (DIAGNOSTICS)
1. Homophily Ratio (Tỷ lệ đồng chất): 0.4884
   -> (Homophily < 0.5 nghĩa là đồ thị mang tính dị chất (Heterophilic). Thông tin giữa các class khác nhau bị trộn lẫn gây nhiễu GNN).

2. Tổng số Edges: 368,134
   -> Cạnh cùng lớp (Safe): 179,792 (48.84%)
   -> Cạnh khác lớp (Poison): 188,342 (51.16%)

3. Phân phối Node Degree (Số kết nối của 1 Node):
   -> Trung bình: 36.81
   -> Cao nhất (Hub): 40.00
   -> Thấp nhất: 17.00
   -> Số Node bị cô lập (Degree=0): 0

4. Thống kê giá trị Feature (x) thô:
   -> Giá trị Min: -5.1993
   -> Giá trị Max: 10.0000
   -> Giá trị Mean: -1.9608


Dynamic IDS Gemini Web gợi ý

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool

class FlowNode_Dynamic_IDS(nn.Module):
    def __init__(self, num_node_features, num_classes, hidden_dim=128):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # 1. SPATIAL MODULE (Trọng số cạnh động bằng Attention)
        self.input_norm = nn.LayerNorm(num_node_features)
        self.linear_in = nn.Linear(num_node_features, hidden_dim)
        
        # GATConv tự động tạo Dynamic Edge Weights
        self.gat1 = GATConv(hidden_dim, hidden_dim // 2, heads=2, concat=True)
        self.norm1 = nn.LayerNorm(hidden_dim)
        
        self.gat2 = GATConv(hidden_dim, hidden_dim, heads=1, concat=False)
        self.norm2 = nn.LayerNorm(hidden_dim)
        
        # 2. TEMPORAL MODULE (Theo dõi trạng thái toàn mạng)
        # GRU Cell nhận đầu vào là trạng thái mạng hiện tại và hidden state trước đó
        self.gru = nn.GRUCell(input_size=hidden_dim, hidden_size=hidden_dim)
        
        # 3. CLASSIFIER MODULE
        self.dropout = nn.Dropout(p=0.3)
        # Phân loại dựa trên: Đặc trưng luồng (hidden_dim) + Lịch sử mạng (hidden_dim)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, edge_index, batch_index, h_t_minus_1=None):
        """
        x: Đặc trưng của các flows (Nodes)
        edge_index: Kết nối giữa các flows chia sẻ IP
        batch_index: Tensor chỉ định mỗi node thuộc đồ thị nào trong batch (dùng cho pooling)
        h_t_minus_1: Trạng thái ngữ cảnh mạng từ timestep trước (Hidden state của GRU)
        """
        # ==========================================
        # BƯỚC 1: HỌC KHÔNG GIAN VÀ TRỌNG SỐ ĐỘNG
        # ==========================================
        x_node = self.input_norm(x)
        x_node = self.linear_in(x_node)
        
        x_node = self.gat1(x_node, edge_index)
        x_node = self.norm1(x_node)
        x_node = F.elu(x_node) # GAT thường dùng ELU
        x_node = self.dropout(x_node)
        
        x_node = self.gat2(x_node, edge_index)
        x_node = self.norm2(x_node)
        x_node = F.elu(x_node)
        
        # ==========================================
        # BƯỚC 2: HỌC THỜI GIAN (GLOBAL MEMORY)
        # ==========================================
        # Gom các flows lại để xem mạng đang bình thường hay đang bị tấn công
        # batch_index mặc định là torch.zeros(x.size(0)) nếu truyền từng chunk 1
        current_network_state = global_mean_pool(x_node, batch_index)
        
        # Khởi tạo hidden state nếu là chunk đầu tiên của chuỗi thời gian
        if h_t_minus_1 is None:
            h_t_minus_1 = torch.zeros(current_network_state.size(0), self.hidden_dim, device=x.device)
            
        # Cập nhật trí nhớ của mô hình
        h_t = self.gru(current_network_state, h_t_minus_1)
        
        # ==========================================
        # BƯỚC 3: KẾT HỢP VÀ PHÂN LOẠI
        # ==========================================
        # Copy "Trí nhớ toàn mạng" (h_t) phát cho từng Flow tương ứng
        # Ví dụ: Nếu mạng đang có dấu hiệu bất thường, từng flow nhỏ cũng sẽ cảnh giác hơn
        h_t_broadcasted = h_t[batch_index] 
        
        # Nối đặc trưng của Node hiện tại với Lịch sử mạng
        final_features = torch.cat([x_node, h_t_broadcasted], dim=1)
        final_features = self.dropout(final_features)
        
        out = self.classifier(final_features)
        
        # Trả về cả prediction và state mới để truyền cho chunk tiếp theo
        return out, h_t

In [24]:
def train_flow_node_dynamic_ids(model, train_snapshots, valid_snapshots, device, class_weights, num_epochs=100):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4) 
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)
    
    best_val_loss = float('inf')
    early_stop_count = 0
    patience = 15
    
    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0.0
        all_train_preds, all_train_labels = [], []
        total_train_nodes = sum(g.num_nodes for g in train_snapshots)
        
        # 1. KHỞI TẠO BỘ NHỚ THỜI GIAN CHO TẬP TRAIN
        h_t = None 
        
        for g in train_snapshots:
            if g.num_nodes == 0: continue
            g = g.to(device)
            
            optimizer.zero_grad(set_to_none=True)
            
            # Tạo batch_index: Tất cả nodes trong snapshot này thuộc cùng 1 đồ thị (batch 0)
            batch_index = torch.zeros(g.num_nodes, dtype=torch.long, device=device)
            
            # 2. ĐƯA DỮ LIỆU & BỘ NHỚ VÀO MÔ HÌNH
            logits, h_t = model(g.x, g.edge_index, batch_index, h_t)
            
            loss = criterion(logits, g.y)
            loss.backward()
            
            # Clip gradient trực tiếp trên từng đồ thị để tránh Gradient Explosion
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            # 3. QUAN TRỌNG: TÁCH BỘ NHỚ KHỎI ĐỒ THỊ TÍNH TOÁN (TBPTT)
            # Giữ lại giá trị (value) của h_t để truyền cho chunk sau, nhưng xóa lịch sử đạo hàm để không tràn RAM
            h_t = h_t.detach()
            
            total_train_loss += (loss.item() * g.num_nodes)
            preds = logits.argmax(dim=1)
            all_train_preds.extend(preds.cpu().tolist())
            all_train_labels.extend(g.y.cpu().tolist())

        train_f1 = f1_score(all_train_labels, all_train_preds, average='macro', zero_division=0)
        avg_train_loss = total_train_loss / max(1, total_train_nodes)
        
        # --- VALIDATION ---
        model.eval()
        total_val_loss = 0.0
        all_val_preds, all_val_labels = [], []
        total_val_nodes = sum(g.num_nodes for g in valid_snapshots)
        
        # Reset bộ nhớ cho chuỗi Validation
        h_t_val = None 
        
        with torch.no_grad():
            for g in valid_snapshots:
                if g.num_nodes == 0: continue
                g = g.to(device)
                
                batch_index = torch.zeros(g.num_nodes, dtype=torch.long, device=device)
                
                # Validation cũng phải chạy tuần tự và truyền h_t_val
                logits, h_t_val = model(g.x, g.edge_index, batch_index, h_t_val)
                loss = criterion(logits, g.y)
                
                total_val_loss += loss.item() * g.num_nodes
                preds = logits.argmax(dim=1)
                all_val_preds.extend(preds.cpu().tolist())
                all_val_labels.extend(g.y.cpu().tolist())
                    
        val_f1 = f1_score(all_val_labels, all_val_preds, average='macro', zero_division=0)
        avg_val_loss = total_val_loss / max(1, total_val_nodes)
        
        scheduler.step(avg_val_loss)
        print(f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f} (F1: {train_f1:.4f}) | Val Loss: {avg_val_loss:.4f} (F1: {val_f1:.4f})")
              
        if avg_val_loss < best_val_loss and not np.isnan(avg_val_loss):
            best_val_loss = avg_val_loss
            early_stop_count = 0
            torch.save(model.state_dict(), "SOTA_DynamicIDS_FlowNode_Best.pth")
        else:
            early_stop_count += 1
            if early_stop_count >= patience:
                print(f"=== Đã kích hoạt Early Stopping tại Epoch {epoch+1}! ===")
                break

    # =================================================================================
    # KIỂM THỬ TRÊN TẬP TEST (TESTING)
    # =================================================================================
    print("\n--- KIỂM THỬ TRÊN TẬP TEST ---")
    try:
        model.load_state_dict(torch.load("SOTA_DynamicIDS_FlowNode_Best.pth", map_location=device))
    except:
        print("Không tải được file Weight, giữ nguyên Weight hiện tại.")
    model.eval()

    all_test_preds, all_test_labels = [], []
    h_t_test = None # Reset bộ nhớ cho chuỗi Test

    with torch.no_grad():
        for g in test_snapshots:
            if g.num_nodes == 0: continue
            g = g.to(device)
            
            batch_index = torch.zeros(g.num_nodes, dtype=torch.long, device=device)
            
            # Test cũng truyền lịch sử
            logits, h_t_test = model(g.x, g.edge_index, batch_index, h_t_test)
            preds = logits.argmax(dim=1)
            
            all_test_preds.extend(preds.cpu().tolist())
            all_test_labels.extend(g.y.cpu().tolist())

    from sklearn.metrics import classification_report
    target_names = [f"Lớp {i}" for i in range(num_classes)]
    print("\nBÁO CÁO PHÂN LOẠI (CLASSIFICATION REPORT) TRÊN TẬP TEST:")
    print(classification_report(all_test_labels, all_test_preds, target_names=target_names, zero_division=0))

In [25]:
# =================================================================================
# 4. KHỞI TẠO MÔ HÌNH VÀ BẮT ĐẦU HUẤN LUYỆN (DYNAMIC IDS)
# =================================================================================
# Cài đặt tham số chiều ẩn (có thể tinh chỉnh tùy vào sức mạnh GPU của bạn)
HIDDEN_DIM = 128 
NUM_EPOCHS = 400

# Khởi tạo mô hình Spatio-Temporal (GAT + GRU)
model = FlowNode_Dynamic_IDS(
    num_node_features=num_node_features, 
    num_classes=num_classes, 
    hidden_dim=HIDDEN_DIM
).to(device)

print(f"\n--- BẮT ĐẦU HUẤN LUYỆN DYNAMIC IDS ---")
print(f"Kiến trúc: Flow-as-Node | Spatial: GAT | Temporal: GRU")
print(f"Device: {device} | Hidden Dim: {HIDDEN_DIM} | Epochs: {NUM_EPOCHS}")
print("-" * 60)

# Gọi hàm huấn luyện chuỗi thời gian
train_flow_node_dynamic_ids(
    model=model, 
    train_snapshots=train_snapshots, 
    valid_snapshots=valid_snapshots, 
    device=device, 
    class_weights=class_weights_tensor, 
    num_epochs=NUM_EPOCHS
)

# Lưu ý: Phần code Testing (đánh giá trên tập test) 
# đã được tôi tích hợp sẵn vào cuối hàm train_flow_node_dynamic_ids ở câu trả lời trước.
# Nên khi chạy xong hàm train này, nó sẽ tự động load best_weight và in ra Classification Report luôn.


--- BẮT ĐẦU HUẤN LUYỆN DYNAMIC IDS ---
Kiến trúc: Flow-as-Node | Spatial: GAT | Temporal: GRU
Device: cuda | Hidden Dim: 128 | Epochs: 400
------------------------------------------------------------


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 001 | Train Loss: 1.0970 (F1: 0.3162) | Val Loss: 2.6293 (F1: 0.1373)
Epoch 002 | Train Loss: 1.5046 (F1: 0.2534) | Val Loss: 2.1108 (F1: 0.1415)
Epoch 003 | Train Loss: 1.3986 (F1: 0.2417) | Val Loss: 2.1362 (F1: 0.1458)
Epoch 004 | Train Loss: 1.5244 (F1: 0.1688) | Val Loss: 1.8198 (F1: 0.1402)
Epoch 005 | Train Loss: 1.0528 (F1: 0.2495) | Val Loss: 1.7856 (F1: 0.1565)
Epoch 006 | Train Loss: 1.8532 (F1: 0.1657) | Val Loss: 1.1957 (F1: 0.1934)
Epoch 007 | Train Loss: 1.2185 (F1: 0.1934) | Val Loss: 1.1445 (F1: 0.2687)
Epoch 008 | Train Loss: 1.0198 (F1: 0.2287) | Val Loss: 1.1314 (F1: 0.1929)
Epoch 009 | Train Loss: 0.9274 (F1: 0.2700) | Val Loss: 0.8424 (F1: 0.2024)
Epoch 010 | Train Loss: 0.8271 (F1: 0.2566) | Val Loss: 0.6625 (F1: 0.3558)
Epoch 011 | Train Loss: 0.6514 (F1: 0.2630) | Val Loss: 0.7570 (F1: 0.2320)
Epoch 012 | Train Loss: 0.8253 (F1: 0.3785) | Val Loss: 0.7164 (F1: 0.4809)
Epoch 013 | Train Loss: 0.9123 (F1: 0.3535) | Val Loss: 0.6537 (F1: 0.2143)
Epoch 014 | 

C:\Users\Admin\AppData\Local\Temp\ipykernel_8380\3829479881.py:96: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("SOTA_DynamicIDS_FlowNode_B


BÁO CÁO PHÂN LOẠI (CLASSIFICATION REPORT) TRÊN TẬP TEST:
              precision    recall  f1-score   support

       Lớp 0       0.99      0.83      0.91     19846
       Lớp 1       0.95      0.05      0.09    484077
       Lớp 2       0.00      0.00      0.00      2515
       Lớp 3       0.91      0.97      0.94     36253
       Lớp 4       0.49      0.95      0.65     18979
       Lớp 5       0.71      0.93      0.80      2451
       Lớp 6       1.00      0.39      0.56     11847
       Lớp 7       0.91      1.00      0.95    107436
       Lớp 8       0.22      0.11      0.14     16746
       Lớp 9       0.06      0.74      0.11     41523
      Lớp 10       1.00      0.98      0.99     18567

    accuracy                           0.34    760240
   macro avg       0.66      0.63      0.56    760240
weighted avg       0.87      0.34      0.32    760240

